# Tutorial 4 — LangGraph: State, Cycles and Agents


By the end you will be able to:

- express a workflow as a **graph over a shared state**, and say what nodes and edges each do;
- route at runtime with **conditional edges**, including edges that point backwards;
- build a **tool-calling agent** that thinks, acts, observes and thinks again;
- replace a hand-built agent with **`create_agent`**, and say what the shortcut hides;
- combine agents as a **pipeline**, a **supervisor** or a set of **handoffs**, and say who decides in each;
- explain why `add_messages` is required, and what breaks without it;
- choose between a chain and a graph for a given problem, with a reason.

## 0. Mental model: a graph is a state machine

<img src="images/langgraph_cards/A_graph_is_a_state_machine.png" alt="A graph is a state machine card: a shared State board holds messages, attempts and result; nodes read the state and return updates, edges choose what runs next, and START, agent, tools, agent, END form the loop. State remembers, nodes work, edges route." width="650">

LangGraph replaces the pipe with a **graph over a shared state**. Nodes read the state and write to it;
edges decide what runs next, and may be conditional or cyclic:

```text
        ┌───────────────────┐
        ▼                   │
START → agent → (decide) ───┘ → END
          │        │
          └→ tools ┘
```

Three pieces, and everything in this notebook is built from them:

| Piece | What it is |
|---|---|
| **State** | A `TypedDict` holding everything the workflow knows so far — the shared whiteboard. |
| **Node** | A function `state → state`. Does work, returns an updated state. |
| **Edge** | What runs next. Fixed, or chosen at runtime by a function reading the state. |

The practical rule: **a chain if the path is known in advance, a graph if the model decides the path.** An
agent that loops "think → call a tool → look at the result → think again" is a cycle, and a cycle is not
something a pipeline can express.

LangGraph is part of the LangChain ecosystem, not a replacement for it: the models, prompts and chains from
tutorial 3 are exactly what goes *inside* the nodes.

## 1. Setup

<img src="images/langgraph_cards/Setup_give_the_graph_its_tools.png" alt="Setup card: load_dotenv reads GROQ_API_KEY from .env, ChatGroq with temperature=0 talks to the Groq API, and model.bind_tools attaches the multiply and current_time tools so the model can return tool_calls requests. Bind tools now, the graph executes them later." width="650">

Same environment as tutorial 3, plus `langgraph`:

```bash
pip install "langchain>=1.0" langchain-core langchain-groq python-dotenv langgraph
```

and the same `.env` beside this notebook:

```text
GROQ_API_KEY=your-key-here
```

The cell below rebuilds the model and the two tools from tutorial 3, section 8, so this notebook stands on
its own.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found — see tutorial 3, section 1."

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.tools import tool

model = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together."""
    return a * b

@tool
def current_time() -> str:
    """Return the current local time as HH:MM."""
    from datetime import datetime
    return datetime.now().strftime("%H:%M")

tools = [multiply, current_time]
model_with_tools = model.bind_tools(tools)

print("model and", len(tools), "tools ready")

### Code walkthrough — what was carried over

Nothing here is new. `ChatGroq` is the same client, `@tool` the same decorator, and **`.bind_tools(tools)`**
the same call that returns a model advertising them.

Worth re-reading, because the rest of the notebook depends on it: the model **never executes anything**. It
emits a structured request — "call `multiply` with `a=17, b=23`" — on the `tool_calls` attribute of its
reply, and your code decides whether to honour it. That boundary is the security model, and the loop you
build in section 4 is what sits on top of it.

## 2. State, nodes, edges

<img src="images/langgraph_cards/State_nodes_edges.png" alt="State, nodes, edges card: GreetState starts with name Maryam and age 30; greet returns only the message key, add_age extends it, and the graph runs START, greet, add_age, END, built with StateGraph, add_node, add_edge, compile and invoke. Return changed keys, keep the rest." width="650">

A graph is built from the three pieces named in section 0. Start with a purely sequential one, so the
machinery is visible before any branching is added.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class GreetState(TypedDict):
    name: str
    age: int
    message: str

def greet(state: GreetState) -> GreetState:
    return {"message": f"Hello, {state['name']}!"}

def add_age(state: GreetState) -> GreetState:
    return {"message": f"{state['message']} You are {state['age']} years old."}

builder = StateGraph(GreetState)
builder.add_node("greet", greet)
builder.add_node("add_age", add_age)

builder.add_edge(START, "greet")
builder.add_edge("greet", "add_age")
builder.add_edge("add_age", END)

graph = builder.compile()

print(graph.invoke({"name": "Maryam", "age": 30})["message"])

### Code walkthrough — state in, partial state out

**`class GreetState(TypedDict)`** declares the shared whiteboard. It is a type annotation, not a container —
LangGraph reads it to know which keys exist.

**A node returns only what it changed.** `greet` returns `{"message": ...}` and says nothing about `name` or
`age`; LangGraph merges that into the running state. Returning the whole state also works, but partial
updates are clearer and are what you will see in real code.

**`START` and `END`** are sentinels marking entry and exit. Every graph needs a path from one to the other.

**`.compile()`** validates the structure — unreachable nodes, missing edges — and returns something with
`.invoke()`. A compiled graph is itself a runnable, so it can sit inside an LCEL chain like any other step.

Run `graph.get_graph().draw_mermaid()` to print a diagram of the structure, which is genuinely useful once
graphs have more than three nodes.

## 3. Conditional edges and loops

<img src="images/langgraph_cards/Conditional_edges_retry_or_stop.png" alt="Conditional edges card: setup, then guess a random number from 1 to 10; should_continue routes to found when the guess equals the target 7, give_up when attempts reach 8, otherwise retry back to guess, using a path map from labels to nodes. Every loop needs a limit." width="650">

The reason to reach for a graph is this section. A **conditional edge** calls a function that inspects the
state and returns the *name* of the next step, which means the path is decided at runtime — and may lead
backwards.

In [ ]:
import random
from typing import List

class GameState(TypedDict):
    guesses: List[int]
    target: int
    attempts: int

def setup(state: GameState) -> GameState:
    return {"guesses": [], "attempts": 0, "target": 7}

def guess(state: GameState) -> GameState:
    return {
        "guesses": state["guesses"] + [random.randint(1, 10)],
        "attempts": state["attempts"] + 1,
    }

def should_continue(state: GameState) -> str:
    if state["guesses"][-1] == state["target"]:
        return "found"
    if state["attempts"] >= 8:
        return "give_up"
    return "retry"

builder = StateGraph(GameState)
builder.add_node("setup", setup)
builder.add_node("guess", guess)

builder.add_edge(START, "setup")
builder.add_edge("setup", "guess")
builder.add_conditional_edges(
    "guess",
    should_continue,
    {"retry": "guess", "found": END, "give_up": END},
)

game = builder.compile()
final = game.invoke({"guesses": [], "target": 0, "attempts": 0})

print("guesses:", final["guesses"])
print("attempts:", final["attempts"])
print("found" if final["guesses"][-1] == final["target"] else "gave up")

### Code walkthrough — the cycle

**`add_conditional_edges(source, router, path_map)`** takes three things: the node the edge leaves, a
function that reads state and returns a **string label**, and a map from labels to destination nodes.

Keeping labels separate from node names is deliberate. The router expresses *a decision* ("retry") while the
map expresses *wiring* ("retry means go back to `guess`"), so you can rewire without touching the logic.

**`"retry": "guess"` points a node back at itself.** That cycle is precisely what a chain cannot express, and
why agents need graphs.

Any loop needs a termination condition that does not depend on the model behaving. Here it is
`attempts >= 8`. In an agent it is a maximum number of tool-calling rounds. Without one, a model that keeps
requesting tools will spin until your budget runs out — the standard failure mode of a first agent.

## 4. A tool-calling agent

<img src="images/langgraph_cards/The_agent_loop.png" alt="Agent loop card: for &#x27;What is 17 x 23, and what time is it?&#x27; the model decides whether tool_calls are needed; if yes, ToolNode runs the functions and ToolMessage results return to the model; if no, the model gives the final answer, with an example transcript." width="650">

Now combine tutorial 3's tools with section 3's cycle. The cycle is: the model thinks; if it asked for tools, run them and return to
the model; otherwise stop. This pattern is called **ReAct** — reason and act.

In [ ]:
from typing import Annotated, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

SYSTEM = SystemMessage(
    "You are a helpful assistant. Use the provided tools when they are relevant, "
    "and answer directly when they are not."
)

def call_model(state: AgentState) -> AgentState:
    return {"messages": [model_with_tools.invoke([SYSTEM] + list(state["messages"]))]}

def needs_tools(state: AgentState) -> str:
    return "tools" if state["messages"][-1].tool_calls else "done"

builder = StateGraph(AgentState)
builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", needs_tools, {"tools": "tools", "done": END})
builder.add_edge("tools", "agent")

agent = builder.compile()

result = agent.invoke({"messages": [HumanMessage("What is 17 times 23, and what time is it?")]})

for m in result["messages"]:
    kind = type(m).__name__
    body = m.content or f"→ calling {[c['name'] for c in m.tool_calls]}"
    print(f"{kind:>12}: {body}")

### Code walkthrough — the agent loop

<img src="images/langgraph_cards/Add_messages_keep_the_conversation.png" alt="add_messages card: by default a node&#x27;s return replaces the messages list and earlier context is lost; with Annotated[Sequence[BaseMessage], add_messages] new messages are appended to the history. Return new messages, preserve the conversation." width="650">

**`Annotated[Sequence[BaseMessage], add_messages]`** is the one piece of genuinely new syntax, and it is
important. By default a node's return value *replaces* a state key. `add_messages` changes that to
**append**, so each node adds to the conversation rather than overwriting it. Forget this annotation and your
agent loses its history every step — the single most common LangGraph bug.

**`ToolNode(tools)`** is prebuilt: it reads `tool_calls` off the last message, runs the matching functions,
and appends a `ToolMessage` per result. You could write it yourself; there is no reason to.

**`needs_tools`** is the whole control flow. If the last message carries `tool_calls`, go and run them;
otherwise the model produced a final answer, so stop.

**`add_edge("tools", "agent")`** closes the cycle. Results go back to the model, which now sees what the
tools returned and decides again — possibly requesting more tools, possibly answering. The printed transcript
shows the full trace: the request, the results, then the answer.

Compare this to tutorial 3, section 8, where the model asked for a tool and nothing happened. The graph is what turns a
request into an action and back into a conversation.

### Exercise 4.1

Add a `search_wikipedia` tool (`pip install wikipedia`, then call `wikipedia.summary(query, sentences=2)`)
and ask something requiring both search and arithmetic — "How tall is the Eiffel Tower in metres, and what is
that in feet?" Watch the message trace: does the model call the tools in one round or two?

Then add a guard: count rounds in the state and force `END` after three. What does the agent do when a tool
keeps returning something it cannot use?

## 5. The shortcut: `create_agent`

Section 4 built the ReAct loop by hand, so you know what is inside it. The loop is common enough that LangChain
ships it as one function. **`create_agent`** (LangChain 1.0 and later) builds the same graph: a model node, a
tools node, and the conditional edge between them.

In [ ]:
from langchain.agents import create_agent

LIMITS = {"recursion_limit": 15}

def show_trace(messages):
    for m in messages:
        kind = m.name or type(m).__name__
        calls = getattr(m, "tool_calls", None)
        body = f"→ calling {[c['name'] for c in calls]}" if calls else m.content
        print(f"{kind:>16}: {body}")

react_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM.content,
    name="assistant",
)

result = react_agent.invoke(
    {"messages": [HumanMessage("What is 17 times 23, and what time is it?")]},
    config=LIMITS,
)

show_trace(result["messages"])

### Code walkthrough: what the one call replaced

| Section 4, by hand | `create_agent` |
|---|---|
| `AgentState` with `add_messages` | Built in |
| `model.bind_tools(tools)` | Done for you, so pass the plain `model` |
| `call_model` with `[SYSTEM] + messages` | `system_prompt=...` |
| `ToolNode(tools)` | Built in |
| `needs_tools` and the conditional edge | Built in |
| `.compile()` | Already compiled |

**The return value is a compiled graph**, the same kind of object as `agent` in section 4. It has `.invoke()`,
and it can be added to another graph as a node. Section 6 depends on this.

**`recursion_limit`** is the termination condition from section 3. It caps the number of graph steps and raises
`GraphRecursionError` when exceeded, so a model that keeps requesting tools cannot spin forever.

**`name="assistant"`** labels the agent's replies, which matters as soon as several agents share a transcript.

Older tutorials use `create_react_agent` from `langgraph.prebuilt`. It is deprecated in favour of this function.

Use `create_agent` when the standard loop is what you want, and write the graph yourself when you need a
different shape.

## 6. Multi-agent systems with `create_agent`

One agent with many tools and a long prompt gets worse as both grow. Splitting the work gives each agent a short
prompt, a few tools, and a context holding only its own task. The price is more model calls, so more latency and
cost: **stay with one agent until it visibly struggles.**

Because `create_agent` returns a graph, there are three natural ways to combine agents. They differ in who decides
what runs next:

| Pattern | Who decides | Agents are used as |
|---|---|---|
| **Pipeline** | You, in the graph's edges | Nodes |
| **Supervisor** | One coordinating agent | Tools |
| **Handoffs** | Whichever agent is active | Nodes that transfer control |

The first two share the tool below, plus `multiply` from section 1. The figures are rounded examples.

In [ ]:
from langchain_core.messages import ToolMessage

POPULATION = {"tehran": 9_000_000, "paris": 2_100_000, "tokyo": 14_000_000}

@tool
def get_population(city: str) -> int:
    """Return the approximate population of a city, or 0 if it is unknown."""
    return POPULATION.get(city.lower(), 0)

def ask(agent, prompt):
    return agent.invoke({"messages": [HumanMessage(prompt)]}, config=LIMITS)

QUESTION = "What is three times the population of Paris?"

### 6.1 Pipeline: agents as nodes

The path is known, so you write it as edges. A researcher collects facts, a writer drafts, and a critic either
approves or sends the draft back. The backwards edge is section 3's cycle, with agents as the nodes:

```text
START → research → write → review → END
                     ▲        │
                     └────────┘ (not approved, revisions left)
```

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy

class Review(BaseModel):
    approved: bool = Field(description="True if the paragraph is correct and answers the question.")
    feedback: str = Field(description="What to fix, or an empty string if approved.")

researcher = create_agent(
    model,
    tools=[get_population, multiply],
    system_prompt="You are a researcher. Use your tools to collect the facts the question needs. "
                  "Reply with short bullet-point notes only.",
    name="researcher",
)
writer = create_agent(
    model,
    system_prompt="You are a writer. Turn the notes into one clear paragraph. Use only facts from "
                  "the notes, and apply any editor feedback you are given.",
    name="writer",
)
critic = create_agent(
    model,
    system_prompt="You are a strict editor. Approve the paragraph only if it answers the question "
                  "and every number in it appears in the notes.",
    response_format=ToolStrategy(Review),
    name="critic",
)

class ReportState(TypedDict):
    question: str
    notes: str
    report: str
    approved: bool
    feedback: str
    revisions: int

def research(state: ReportState) -> ReportState:
    return {"notes": ask(researcher, state["question"])["messages"][-1].content}

def write(state: ReportState) -> ReportState:
    prompt = (f"Question: {state['question']}\nNotes: {state['notes']}\n"
              f"Editor feedback: {state['feedback']}")
    return {"report": ask(writer, prompt)["messages"][-1].content,
            "revisions": state["revisions"] + 1}

def review(state: ReportState) -> ReportState:
    prompt = (f"Question: {state['question']}\nNotes: {state['notes']}\n"
              f"Paragraph: {state['report']}")
    verdict = ask(critic, prompt)["structured_response"]
    return {"approved": verdict.approved, "feedback": verdict.feedback}

def after_review(state: ReportState) -> str:
    if state["approved"] or state["revisions"] >= 3:
        return "done"
    return "revise"

builder = StateGraph(ReportState)
builder.add_node("research", research)
builder.add_node("write", write)
builder.add_node("review", review)

builder.add_edge(START, "research")
builder.add_edge("research", "write")
builder.add_edge("write", "review")
builder.add_conditional_edges("review", after_review, {"revise": "write", "done": END})

pipeline = builder.compile()
final = pipeline.invoke({"question": QUESTION, "feedback": "none", "revisions": 0})

print("notes:", final["notes"])
print("report:", final["report"])
print("approved:", final["approved"], "after", final["revisions"], "draft(s)")

### Code walkthrough: agents inside a graph

**Each node wraps an agent in a small function.** The wrapper chooses exactly what the agent sees and which key
its answer lands in. The writer never sees the researcher's tool calls, only the finished notes, which keeps its
context short.

**`response_format=ToolStrategy(Review)`** makes the critic return a validated `Review` object, found under
`result["structured_response"]`. The router reads a boolean instead of searching the reply for the word
"approved", so a change in the model's phrasing cannot break the control flow.

**`state["revisions"] >= 3`** is the limit that does not depend on the model behaving, as section 3 requires. A
critic that never approves still terminates, so check `approved` in the output rather than assuming it.

### 6.2 Supervisor: agents as tools

Now the path depends on the request, so a model should choose it. Wrap each worker in a `@tool` function and give
those tools to a coordinating agent. The supervisor is an ordinary ReAct agent whose tools happen to be agents:

```text
              ┌→ ask_data_agent → data_agent ┐
supervisor ───┤                              ├→ supervisor → answer
              └→ ask_math_agent → math_agent ┘
```

In [ ]:
data_agent = create_agent(
    model,
    tools=[get_population],
    system_prompt="You look up city populations. Reply with the figure and nothing else.",
    name="data_agent",
)
math_agent = create_agent(
    model,
    tools=[multiply],
    system_prompt="You do arithmetic with your tools. Reply with the result and nothing else.",
    name="math_agent",
)

@tool
def ask_data_agent(task: str) -> str:
    """Look up a city's population. Describe the lookup in plain language."""
    return ask(data_agent, task)["messages"][-1].content

@tool
def ask_math_agent(task: str) -> str:
    """Do arithmetic. Include every number needed, since this agent sees nothing else."""
    return ask(math_agent, task)["messages"][-1].content

supervisor = create_agent(
    model,
    tools=[ask_data_agent, ask_math_agent],
    system_prompt="You coordinate two specialists. Split the question into steps, delegate each "
                  "step, then give the final answer. Never compute or recall figures yourself.",
    name="supervisor",
)

result = ask(supervisor, QUESTION)
show_trace(result["messages"])

### Code walkthrough: the tool boundary

**The wrapper is the whole trick.** `ask_math_agent` runs a complete agent loop and returns one string. To the
supervisor it looks like `multiply` did in section 4.

**Context is isolated in both directions.** A worker sees only the `task` string, so that string must be
self-contained, which is why the docstring says so. The supervisor sees only the worker's final reply, and none of
its tool calls. That is why the printed trace shows the delegations rather than every inner step.

**Docstrings are now routing logic.** The supervisor picks a worker from the tool's name and docstring, exactly as
any model picks a tool, so a vague docstring produces bad delegation.

Workers answer the supervisor and never the user. That makes this pattern easy to control, and it means every
request pays for the supervisor's extra model calls.

### 6.3 Handoffs: agents pass control

In a handoff there is no coordinator. The active agent decides that someone else should take over, and that agent
then talks directly to the user with the full conversation in front of it. This suits a helpdesk, where a triage
agent passes you to a specialist:

```text
START → triage ──transfer_to_grades──→ grades → END
           │
           └─────transfer_to_exams───→ exams  → END
```

In [ ]:
from langchain.tools import ToolRuntime
from langgraph.graph import MessagesState
from langgraph.types import Command

GRADES = {"s101": 18.5, "s102": 16.0}
EXAM_DATES = {"llm": "2026-12-20", "nlp": "2026-12-27"}

@tool
def get_grade(student_id: str) -> str:
    """Return a student's midterm grade, given an id such as s101."""
    return str(GRADES.get(student_id.lower(), "no grade on record"))

@tool
def get_exam_date(course: str) -> str:
    """Return the final exam date for a course, given a name such as llm."""
    return EXAM_DATES.get(course.lower(), "no exam on record")

def make_handoff(target: str, description: str):
    @tool(f"transfer_to_{target}", description=description)
    def handoff(runtime: ToolRuntime) -> Command:
        request = next(m for m in reversed(runtime.state["messages"]) if isinstance(m, AIMessage))
        receipt = ToolMessage(f"Transferred to {target}.", tool_call_id=runtime.tool_call_id)
        return Command(goto=target, update={"messages": [request, receipt]}, graph=Command.PARENT)
    return handoff

triage = create_agent(
    model,
    tools=[
        make_handoff("grades", "Transfer the student to the grades specialist."),
        make_handoff("exams", "Transfer the student to the exam schedule specialist."),
    ],
    system_prompt="You are a course helpdesk. Make exactly one transfer to the right specialist. "
                  "If neither fits, answer briefly yourself.",
    name="triage",
)
grades = create_agent(
    model,
    tools=[get_grade],
    system_prompt="You answer questions about grades. Use your tool, then reply to the student.",
    name="grades",
)
exams = create_agent(
    model,
    tools=[get_exam_date],
    system_prompt="You answer questions about exam dates. Use your tool, then reply to the student.",
    name="exams",
)

builder = StateGraph(MessagesState)
builder.add_node("triage", triage, destinations=("grades", "exams"))
builder.add_node("grades", grades)
builder.add_node("exams", exams)

builder.add_edge(START, "triage")
builder.add_edge("grades", END)
builder.add_edge("exams", END)

helpdesk = builder.compile()

result = ask(helpdesk, "When is the final exam for the LLM course?")
show_trace(result["messages"])

### Code walkthrough: `Command` moves control

**`Command(goto=target, graph=Command.PARENT)`** is the handoff. A tool normally returns a result to its own agent;
this one tells the *parent* graph to leave the current agent and run `target` instead. That is why the agents must
be nodes of a shared parent graph.

**`update={"messages": [request, receipt]}`** keeps the history valid. The AI message requesting the transfer and
the `ToolMessage` answering it must arrive as a pair, because a tool call with no result is rejected by model APIs.
Leaving the pair out is the usual first bug here.

**`ToolRuntime`** is injected by LangChain and hidden from the model, which sees a tool with no arguments. It gives
the tool access to the agent's state and to the id of the current tool call.

**`destinations=(...)`** only tells the diagram where `triage` can go. The real routing happens at runtime through
`Command`. `triage` has no outgoing edge, so if it answers without transferring, the graph ends there.

The system prompt says "exactly one transfer" for a reason: two transfers in a single reply would ask the graph to
go two ways at once.

### Exercise 6.1

1. **Pipeline.** Make the critic impossible to satisfy by editing its prompt. Confirm the graph stops after three
   drafts, then decide what a real application should show the user in that case.
2. **Supervisor.** Change the `ask_math_agent` docstring to just "Math helper." and ask `QUESTION` again. Does the
   supervisor still pass the population to the worker?
3. **Handoffs.** Ask the helpdesk "What is the capital of France?" and read the trace. Then add a `transfer_to_triage`
   tool to `grades` and `exams`, with `"triage"` as a destination. What new loop have you created, and what stops it?
4. **Compare.** Count the model calls each pattern makes for one request. Which would you ship for a question that
   nearly always needs only one specialist?

## 7. Choosing between them

<img src="images/langgraph_cards/Chain_or_graph.png" alt="Chain or graph card: chains (LCEL) suit fixed steps, parallel calls, one-branch choices and standard RAG; graphs (LangGraph) suit repeated tool calls, runtime control, shared agent state and pause and resume. LCEL is data flow, LangGraph is control flow." width="650">

| Situation | Reach for |
|---|---|
| Fixed sequence of steps known in advance | **Chain** |
| Several independent calls on one input | **Chain** with `RunnableParallel` |
| One of N paths, chosen once | **Chain** with `RunnableBranch` |
| Retrieve-then-answer (standard RAG) | **Chain** |
| The model decides what happens next | **Graph** |
| Anything that can repeat a step | **Graph** |
| Multiple cooperating agents, shared state | **Graph** |
| You need to pause, inspect, or resume | **Graph** |

Chains are easier to read, easier to test and easier to debug. **Start with a chain and move to a graph when
you meet a cycle** — not before. A graph built for a pipeline that never branches is just a pipeline with
more ceremony.

The honest summary of the split: LCEL handles *data flow*, LangGraph handles *control flow*. Most real
applications contain both, with chains as the nodes inside a graph.

## Glossary — quick reference

<img src="images/langgraph_cards/LangGraph_pocket_dictionary.png" alt="LangGraph pocket dictionary card: State, Node, Edge, conditional edge, path map, START and END, compile(), add_messages and ToolNode, with the ReAct cycle of reason, act, observe, repeat." width="650">

| Term | Meaning |
|---|---|
| **State** | The `TypedDict` a graph passes between nodes — the shared whiteboard. |
| **Node** | A function taking the state and returning the keys it changed. |
| **Edge** | A fixed connection between nodes. |
| **Conditional edge** | An edge whose destination is chosen at runtime by a function reading the state. |
| **Path map** | The dict translating a router's string labels into destination nodes. |
| **`START` / `END`** | Sentinels marking a graph's entry and exit. |
| **`.compile()`** | Validates the graph and returns a runnable, usable inside an LCEL chain. |
| **`add_messages`** | Annotation making a state key append rather than overwrite. Required for conversations. |
| **`ToolNode`** | Prebuilt node that reads `tool_calls`, runs the functions, and appends the results. |
| **ReAct** | The reason-then-act loop: think, call a tool, observe, think again. |
| **Cycle** | An edge pointing back to an earlier node — the thing a chain cannot express. |
| **`create_agent`** | One call that builds the ReAct loop and returns a compiled graph. Needs LangChain 1.0 or later. |
| **Pipeline** | Agents as graph nodes, with the path fixed by your edges. |
| **Supervisor** | A coordinating agent that calls worker agents wrapped as tools. |
| **Handoff** | An agent transferring control, and the conversation, to another agent. |
| **`Command`** | A return value that updates state and chooses the next node. `graph=Command.PARENT` targets the parent graph. |

---